# 02 CIGNN EMNIST

> EMNIST letter exp

In [ ]:
#| default_exp data

%load_ext autoreload
%autoreload 2

In [ ]:

import os
import pandas as pd
import sys
sys.path.append(os.path.abspath('..'))

import torch
import torchvision
from torchvision.transforms import ToTensor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision.transforms import functional as F
from torchvision.io import read_image, ImageReadMode
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary
import numpy as np
import scipy
from sklearn import metrics
import random

from sklearn.metrics import pairwise_distances, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from scipy.interpolate import CubicSpline

from CIGNN_experimental.unipen_renderer.parser import Parser
from CIGNN_experimental.unipen_renderer.strokes_to_image_converter import Strokes2ImageConverter

import time


In [ ]:
project="CIGNN_GA_online_synth_VGG16_bn"
n_epochs = 10 
n_classes = 10
n_prototypes_per_class = 1
n_samples_per_class = 1024
n_generations = 250
n_genomes = 10_000

dev = "cuda"
env = "prod"


In [ ]:
from dataclasses import dataclass, field

@dataclass
class Sample():
    fitness: float
    origin: int 
    strokes: list[list[tuple[float, float]]] = field(default_factory=list)
    stroke_width: int = 3

In [ ]:
# collect all parameters and initialise wandb project
import matplotlib.pyplot as plt
import torch
import wandb

wandb.init(
    project=project,
    config = {
        "n_epochs": n_epochs,
        "n_generations": n_generations,
        "n_genomes": n_genomes,
        "n_classes": n_classes,
        "n_prototypes_per_class": n_prototypes_per_class,
        "n_samples_per_class": n_samples_per_class,
        "dev" : dev,
        "env" : env
    }
)


def render_samples(samples, true_labels=[], predicted_labels=[], title=""):
    fig, axes = plt.subplots(nrows=1, ncols=len(samples), figsize=(len(samples) * 3, 5))
    axes[0].title.set_text(f"{title} - [true: {true_labels} // pred: {predicted_labels}]")
    
    for idx, sample in enumerate(samples):
        for stroke in sample.strokes:
            arr_stroke = np.array(stroke)
            axes[idx].plot(arr_stroke[:, 0], arr_stroke[:, 1])
            axes[idx].plot(arr_stroke[:, 0], arr_stroke[:, 1], "o")

    
    wandb.log({title: wandb.Image(fig)})
    plt.close()
    


# Read the preprocessed stroke data 
data = np.load("../CIGNN_experimental/unipen_renderer/unipen_strokes.npy", allow_pickle=True).item()
data = {cls: data[cls + 65] for cls in range(n_classes) if (cls + 65) in data.keys()}

# Convert all items to samples because this is expected by the dataset class 
for cls in data.keys():
    data[cls] = [Sample(fitness=1.0, origin=cls, strokes=x, stroke_width=3) for x in data[cls]]

# Split data into data, test so that for each class randomly 80% go into the training set and 20% into the testing set
train_data = []
test_data = []

for cls, samples in data.items():
    train_samples, test_samples = train_test_split(samples, test_size=0.2, train_size=0.8)
    train_data.extend(train_samples)
    test_data.extend(test_samples)

# Choose one prototype per class from the data (training set)
prototypes = []
for cls, samples in data.items():
    # Randomly choose one sample of each class as prototype.
    # For now just use one prototype per class
    rand_idx = np.random.choice(len(samples))  
    prototypes.append(samples[rand_idx])

# Print the prototypes to wandb
render_samples(prototypes, title="Prototypes")

In [ ]:


class UnipenDataset(Dataset):
    
    def __init__(self, population, device=None, image_size=(64, 64)):
        """
        """
        from torchvision.transforms.functional import pil_to_tensor
        
        self.data = []
        s2i = Strokes2ImageConverter()

        if not device:
            self.device = torch.device("cuda")

        for sample in population:
            #img = s2i.strokes_to_image(strokes=sample.strokes, image_size=image_size, smoothing_factor=0.0, stroke_width=np.random.randint(5) + 1).convert("L")
            img = s2i.strokes_to_image(strokes=sample.strokes, image_size=image_size, smoothing_factor=0.0, stroke_width=sample.stroke_width).convert("RGB")
            # Finally, the image needs to be converted to tensors
            tensor = pil_to_tensor(img).to(device)
            self.data.append((tensor, sample.origin))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, label = self.data[idx]
        return img, label



In [ ]:
# Crossover methods

def linear_interpolation(start, end, num_points=3):
    """
    Perform linear interpolation between two points. This method can be used in cases where spline-fitting is impossible when splines have <3 points.
    This method can fail if start == end!

    Args: 
        start: x-y-tuple of the starting point
        end: x-y-tuple of the ending point
        num_points: How many points should be interpolated

    Returns:
        List of interpolated points (which can be used for spline fitting afterwards)
    """

    if start[0] == end[0] and start[1] == end[1]:
        raise ValueError(f"Can not interpolate {start} - {end}. Aborting.")

    xs = np.linspace(start[0], end[0], num=num_points)
    ys = np.linspace(start[1], end[1], num=num_points)

    return list(zip(xs, ys))

def spline_crossover(prototype1, prototype2):
    """
    Perform spline-based crossover between two prototypes.

    Args:
        prototype1: First parent prototype (list of strokes).
        prototype2: Second parent prototype (list of strokes).
        num_points: Number of points to sample from the combined spline.

    Returns:
        offspring: New prototype generated from the crossover.
    """
    offspring = []

    for stroke1, stroke2 in zip(prototype1, prototype2):
    
        if len(stroke1) < 3:
            # If the stroke consists of only a single value we simply return the other stroke
            if len(stroke1) == 1:
                return stroke2
            else:
                # Attempt an interpolation
                stroke1 = linear_interpolation(stroke1[0], stroke1[-1])
        if len(stroke2) < 3:
            # If the stroke consists of only a single value we simply return the other stroke
            if len(stroke2) == 1:
                return stroke1
            else:
                # Attempt an interpolation
                stroke2 = linear_interpolation(stroke2[0], stroke2[-1])
        # Fit cubic splines to both parent strokes
        points1 = np.array(stroke1)
        points2 = np.array(stroke2)
        t1 = np.linspace(0, 1, len(points1))
        t2 = np.linspace(0, 1, len(points2))
        cs_x1 = CubicSpline(t1, points1[:, 0])
        cs_y1 = CubicSpline(t1, points1[:, 1])
        cs_x2 = CubicSpline(t2, points2[:, 0])
        cs_y2 = CubicSpline(t2, points2[:, 1])

        # Combine the splines (e.g., using single-point crossover)
        t_combined = np.linspace(0, 1, (len(stroke1) + len(stroke2)) // 2)
        x_combined = (cs_x1(t_combined) + cs_x2(t_combined)) / 2
        y_combined = (cs_y1(t_combined) + cs_y2(t_combined)) / 2

        # Create the offspring stroke
        offspring_stroke = list(zip(x_combined, y_combined))
        offspring.append(offspring_stroke)

    return offspring

In [ ]:

from CIGNN_experimental.stroke_transforms import random_move, random_move_stroke, smooth_strokes, shorten, prolong, merge_points, skip_points, merge_strokes, split_strokes

def mutate(population: list[Sample], p: float = 0.2, mutation_params={}) -> list[Sample]:
    
    mutated_population = []
    for sample in population: 
        if np.random.rand() < p*100:

            new_strokes = []
            # Randomly choose the mutation to apply
            # mut_idx = np.random.randint(1) + 4
            mut_idx = 0

            if mut_idx == 0:
                new_strokes = random_move(sample.strokes, max_dist=(1,1))
            elif mut_idx == 1:
                new_strokes = random_move_stroke(sample.strokes, max_dist=(3, 3))
            elif mut_idx == 2:
                new_strokes = smooth_strokes(sample.strokes)
            elif mut_idx == 3:
                new_strokes = shorten(sample.strokes)
            elif mut_idx == 4:
                new_strokes = prolong(sample.strokes)
            elif mut_idx == 5:
                new_strokes = merge_points(sample.strokes)
            elif mut_idx == 6:
                new_strokes = skip_points(sample.strokes)
            elif mut_idx == 7:
                new_strokes = merge_strokes(sample.strokes, distance_threshold=1e6)
            elif mut_idx == 8: 
                new_strokes = split_strokes(sample.strokes)
            else:
                new_strokes = sample.strokes

            mutated_population.append( Sample(fitness=sample.fitness, origin=sample.origin, strokes=new_strokes))
            

        else:
            mutated_population.append(sample)
        
    return mutated_population

def crossover(population: list[Sample]):

    probabilities = [sample.fitness for sample in population]
    normalized_probabilities = probabilities / np.sum(probabilities) # These can never be negative
    pairs = np.random.choice(population, size=(len(population), 2), p=normalized_probabilities)

    next_generation: list[Sample] = []
    # At this point we have pairs of shape (pop_size, 2)
    for p1, p2 in pairs:         
        next_generation_strokes = spline_crossover(p1.strokes, p2.strokes)
        next_generation.append(Sample(1.0, p1.origin, next_generation_strokes))
    
    return next_generation    

In [ ]:
from numpy import ndarray
from torch.utils.data import Dataset, DataLoader

from torchvision.transforms.functional import to_pil_image

def next_generation(population: ndarray[Sample]) -> ndarray[Sample]:

    next_gen = np.array([])
    unique_classes = np.unique([sample.origin for sample in population])
        
    for cls in unique_classes:
        population_by_class = list(filter(lambda x: x.origin == cls, population))
        mutated_population = mutate(population_by_class)
        offspring = crossover(mutated_population)
        next_gen = np.concatenate((next_gen, offspring))

    return next_gen



# Create the first generation
# Repeat each prototype for as many times as defined in `n_samples_per_class`, afterwards flatten the list
population = np.array([[Sample(1.0, idx, prototype.strokes)] * n_samples_per_class for idx, prototype in enumerate(prototypes)]).reshape(-1)
new_population = next_generation(population)

ds = UnipenDataset(new_population)
dl = DataLoader(ds, batch_size=16, shuffle=True)



In [ ]:
model = torchvision.models.vgg16_bn(num_classes=n_classes)
device = torch.device(dev)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Save the initial model to re-load the exact parameters 
initial_checkpoint_path = "21_CIGNN_05_model_init.pth"
torch.save({
    'model_state_dict': model.state_dict(),
}, initial_checkpoint_path) 

In [ ]:
# Train the baseline model 
# For the baseline we use all of the training data (after splitting away a validation set)
# Make sure to use the same initial weights for each run, including synth runs, and 
# train the model n-times to obtain proper averages

# Repeat 5 times
run_N = 1 
n_epochs = 5
for run_i in range(run_N):
    
    # Create datasets and split up the full training set to 80:20
    full_train_set = UnipenDataset(train_data, device)
    test_set = UnipenDataset(test_data, device)
    train_set, val_set = torch.utils.data.random_split(full_train_set, [0.8, 0.2])
    train_dl, val_dl = DataLoader(train_set, batch_size=16, shuffle=True), DataLoader(val_set, batch_size=16, shuffle=True)
    test_dl = DataLoader(test_set, batch_size=16, shuffle=True)

    # Init model and optimizer
    # (by loading the initial checkpoint)
    model = torchvision.models.vgg16_bn(num_classes=n_classes)
    model.to(device)
    checkpoint = torch.load(initial_checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)   
    
    # train-val loop
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
    
        # Training phase
        for batch_idx, (images, labels) in enumerate(train_dl, start=1):
            images, labels = images.to(device).to(torch.float), labels.to(device)
            optimizer.zero_grad()  # Clear gradients
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Calculate loss
            loss.backward()  # Backward pass
            optimizer.step()  # Update weights
            running_loss += loss.item()
    
            # Print progress every 10 batches
            if batch_idx % 10 == 0:
                wandb.log({"train_loss": loss.item(), "run": run_i})
                print(f"Epoch [{epoch+1}/{n_epochs}], Batch [{batch_idx}/{len(train_dl)}], Loss: {running_loss/batch_idx:.4f}")
    
        avg_train_loss = running_loss / len(train_dl)
        print(f"Epoch [{epoch+1}/{n_epochs}], Training Loss: {avg_train_loss:.4f}")
    
        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_dl:  # Use a separate validation dataloader
                images, labels = images.to(device).to(torch.float), labels.to(device)
                outputs = model(images)  # Forward pass
                loss = criterion(outputs, labels)  # Calculate validation loss
                val_loss += loss.item()
    
        avg_val_loss = val_loss / len(val_dl)
        print(f"Epoch [{epoch+1}/{n_epochs}], Validation Loss: {avg_val_loss:.4f}")
        wandb.log({"val_loss": loss.item(), "run": run_i})

    model.eval()
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device).to(torch.float), labels.to(device)
            outputs = model(images)  # Forward pass
            predicted = outputs.argmax(dim=1)  # Get predicted class
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            
            # Collect all labels and predictions for metrics calculation
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    # Compute overall accuracy
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    
    # Compute F1 scores per class
    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    f1_per_class = f1_score(all_labels, all_predictions, average=None)
    for i, f1 in enumerate(f1_per_class):
        print(f"F1 Score for Class {i}: {f1:.4f}")
        wandb.log({f"F1_score_class_{i}": f1, "run": run_i})
    
    # Log overall F1 score (macro-averaged)
    f1_macro = f1_score(all_labels, all_predictions, average="macro")
    print(f"Macro-averaged F1 Score: {f1_macro:.4f}")
    wandb.log({"f1_macro": f1_macro, "run": run_i})
    
    # Compute and display the confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_predictions)
    print("Confusion Matrix:")
    print(conf_matrix)
    
    # Log confusion matrix to wandb as a table or heatmap
    wandb.log({"confusion_matrix": wandb.Table(data=conf_matrix.tolist(), columns=[f"Class_{i}" for i in range(len(conf_matrix))])})


In [ ]:
model.eval()
correct = 0
total = 0
all_labels = []
all_predictions = []

test_dl = DataLoader(test_set, batch_size=16, shuffle=True)


with torch.no_grad():
    for images, labels in test_dl:
        images, labels = images.to(device).to(torch.float), labels.to(device)
        outputs = model(images)  # Forward pass
       

        for idx, sample in enumerate(images):
            # print(sample)
            plt.title(f"True: {chr(labels[idx].item() + 65)}  Pred: {chr(outputs.argmax(dim=1)[idx] + 65)}")
            plt.imshow(sample.permute(1, 2, 0).to(torch.int).cpu().numpy())
            plt.show()
        break

In [ ]:


best_val_loss = float('inf')  # Initialize the best validation loss
checkpoint_path = 'best_model_checkpoint.pth'  # Path to save the best model

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0

    # Training phase
    for batch_idx, (images, labels) in enumerate(train_dataloader, start=1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()  # Clear gradients
        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights
        running_loss += loss.item()

        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            wandb.log({"train_loss": loss.item()})
            print(f"Epoch [{epoch+1}/{n_epochs}], Batch [{batch_idx}/{len(train_dataloader)}], Loss: {running_loss/batch_idx:.4f}")

    avg_train_loss = running_loss / len(train_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Training Loss: {avg_train_loss:.4f}")

    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_dataloader:  # Use a separate validation dataloader
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Calculate validation loss
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch [{epoch+1}/{n_epochs}], Validation Loss: {avg_val_loss:.4f}")
    wandb.log({"val_loss": loss.item()})


    # Checkpoint saving
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Best model updated and saved at epoch {epoch+1} with Validation Loss: {avg_val_loss:.4f}")

# Testing loop
model.load_state_dict(torch.load(checkpoint_path))  # Load the best model checkpoint
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)  # Forward pass
        predicted = outputs.argmax(dim=1)  # Get predicted class
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
wandb.log({"test_acc": accuracy})


In [ ]:
# Reference datasetes (without transformations etc)
from sklearn.cluster import KMeans
from collections import Counter


ref_train_data = UnipenCuratedDataset(train_split, data_dir)
ref_train_dataloader = DataLoader(dataset=ref_train_data, batch_size=32, shuffle=True)
ref_val_data = UnipenCuratedDataset(val_split, data_dir)
ref_val_dataloader = DataLoader(dataset=ref_val_data, batch_size=32, shuffle=True) 

# Initialize random genomes along with their fitness scoring 
genomes = [(np.random.rand(6), 0) for _ in range(n_genomes)]

# For each generation of the algorithm ...
for generation in range(n_generations):   
    
    # Keep track of the current generation's fitness score 
    curr_generation = []

    # For each genome, generate a training and validation dataset
    for genome, score in genomes:
        
        start_time = time.time()

        train_dataloader, val_dataloader = generate_dataset(genome=genome, n_samples_train=n_samples_train, n_samples_val=n_samples_val)
        

        # train the reference network with this data
        # Make sure the model is a **new instance**
        vgg_bn = torchvision.models.vgg16_bn(num_classes=93)
        vgg_bn.to(device)
        model = vgg_bn

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.0001)

        # Train the reference network 
        for epoch in range(n_epochs):
            model.train()
            running_loss = 0.0

            # Training phase
            for batch_idx, (images, labels) in enumerate(train_dataloader, start=1):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()  # Clear gradients
                outputs = model(images)  # Forward pass
                loss = criterion(outputs, labels)  # Calculate loss
                loss.backward()  # Backward pass
                optimizer.step()  # Update weights
                running_loss += loss.item()

                # Print progress every 10 batches
                if batch_idx % 10 == 0:
                    wandb.log({"train_loss": loss.item()})


            # Validation phase
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for images, labels in val_dataloader:  # Use a separate validation dataloader
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)  # Forward pass
                    loss = criterion(outputs, labels)  # Calculate validation loss
                    val_loss += loss.item()

            avg_val_loss = val_loss / len(val_dataloader)
            wandb.log({"average val loss": avg_val_loss})


        # Evaluate the clustering metrics of the network
        # To generate the clusters we run a single pass on train+val set using true labels 
        # and the activation at the FCN output layer. It is important to use datasets here wihtout the
        # augmentations and transformations. 
        # The algorithm used a single-pass kNN with n_classes = n_clusters.
        # To evaluate the resulting clusters we use the reference (testing) set. 


        model.eval()

        all_features, all_labels = [], []
        
        combined_dataset = ConcatDataset([ref_train_dataloader.dataset, ref_val_dataloader.dataset])
        combined_dataloader = DataLoader(combined_dataset, batch_size=ref_train_dataloader.batch_size)
        
        activations = None 

        def hook_fn(module, input, output):
            global activations 
            activations = output

        hook = model.classifier[5].register_forward_hook(hook_fn)

        with torch.no_grad():
            for images, labels in combined_dataloader:
                images, labels = images.to(device), labels.to(device)
                model(images)
                
                all_features.append(activations.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        all_features = np.concatenate(all_features, axis=0)
        all_labels = np.concatenate(all_labels, axis=0)


        num_clusters = len(np.unique(all_labels))  # Number of clusters = Number of classes
        
       
        kmeans = KMeans(n_clusters=num_clusters, random_state=42)
        kmeans.fit(all_features)  # Fit on the combined features

        ##                                   SKIP THIS FOR NOW                                  ##
        # Map clusters to classes based on majority vote
        #cluster_to_class = {}
        #for cluster_id in range(num_clusters):
        #    # Get indices of points in the current cluster
        #    cluster_indices = np.where(kmeans.labels_ == cluster_id)[0]
        #    # Find the most common class label in this cluster
        #    majority_class = Counter(all_labels[cluster_indices]).most_common(1)[0][0]
        #    cluster_to_class[cluster_id] = majority_class

        #print(cluster_to_class)
            
        # Evaluate on a reference (test) set
        #ref_features = []
        #ref_labels = []

        ##                                   SIMPLIFICATION                                    ## 
        # Offset the silhouette score by 1 to avoid negative weights in the random.choices() selection
        # during the mutation and crossover steps 
        fitness = metrics.silhouette_score(all_features, all_labels, metric='euclidean') + 1
        
        curr_generation.append((genome, fitness))
        
                
        
        end_time = time.time()
        wandb.log({"fitness": fitness})
        wandb.log({"time per genome [s]": end_time - start_time})
        
        del vgg_bn       
        del train_dataloader
        del val_dataloader
        torch.cuda.empty_cache()

    
    
    # crossover and mutation step for genomes
    genomes = evolve_population(curr_generation)
    
     





In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()